In [ ]:
import pandas as pd
import numpy as np
import time

from scipy.integrate import odeint
from scipy.optimize import minimize

import pickle

from pyabc import ABCSMC, Distribution, RV, MultivariateNormalTransition, QuantileEpsilon

import pyabc
from pyabc import ABCSMC, Distribution, RV
import tempfile

pyabc.settings.set_figure_params('pyabc') 

from DDE_s import *
from Parameters_s import *
import sympy
from jitcdde import y, t, jitcdde

import warnings
warnings.filterwarnings("ignore")

_ = np.random.seed(0)

In [ ]:
def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return np.asarray(with_noise)

In [ ]:
with open('../../Data/Model3/sim_dataset.pkl', 'rb') as handle:
    simulation_dataset = pickle.load(handle)

In [ ]:
param_names = ["beta", "phi_s", "g_rate"]

In [ ]:
all_control_parameters_sym = list(sympy_params.values()) + list(sympy_delays.values())
current_control_parameter_values = [params[k] for k in sympy_params.keys()] + [T_delay[k] for k in sympy_delays.keys()]
param_to_index = {sym: i for i, sym in enumerate(all_control_parameters_sym)}
max_delay_value = max(T_delay.values())

In [ ]:
def sample_prior(n):
    return {
        'beta': np.random.uniform(0.6, 1.2, n),
        'phi_s': np.random.uniform(0.2, 0.6, n),
        'g_rate': np.random.uniform(0.005, 0.01, n),
    }
    
def sampleResult(yy):
    samp_index = [int(i) for i in np.linspace(0, N_t * sample_num_factor, N_t+1)]
    sampled_yy = []
    for t_i in samp_index:
        sampled_yy.append(yy[t_i])
    return sampled_yy


def simulate(parameters):
    beta, phi_s, g_rate = parameters
    
    current_control_parameter_values[param_to_index[sympy_params['beta']]]=beta
    current_control_parameter_values[param_to_index[sympy_params['beta_prime']]]=beta*1.2
    current_control_parameter_values[param_to_index[sympy_params['phi_s_1']]]=phi_s
    current_control_parameter_values[param_to_index[sympy_params['g_beta']]]=g_rate

    DDE = jitcdde(f, control_pars=all_control_parameters_sym, max_delay=max_delay_value)
    DDE.set_integration_parameters(atol=1e-6, rtol=1e-3)
    DDE.constant_past(init_cond)
    DDE.set_parameters(current_control_parameter_values)

    data = [DDE.integrate(time) for time in times]
    yy = sampleResult(data)
    
    return yy

def simulator(sim_par):
    beta = sim_par[0]
    phi_s = sim_par[1]
    g_rate = sim_par[2]

    yy = simulate(parameters=[beta, phi_s, g_rate])
    
    pos = []

    indices = np.arange(0, N_t * sample_num_factor, sample_num_factor)
    
    pos = [(yy[t][states['F_AT_1']] + yy[t][states['F_ST_1']]) * (params['True_P_1']) +
           (yy[t][states['F_AT_2']] + yy[t][states['F_ST_2']]) * (params['True_P_2']) +
           (1 - params["True_N_1"]) * (yy[t][states['F_NT_1']] + yy[t][states['F_AT_3']] +
                                       yy[t][states['F_FT_1']] + yy[t][states['F_GT1']] +
                                       yy[t][states['F_ST_4']] + yy[t][states['F_ST_3']] +
                                       yy[t][states['F_FT_3']]) +
           (1 - params["True_N_2"]) * (yy[t][states['F_NT_2']] + yy[t][states['F_AT_4']] +
                                       yy[t][states['F_FT_2']] + yy[t][states['F_GT2']]) for t in range(N_t)]

    
    neg = [(yy[t][states['F_AT_1']] + yy[t][states['F_ST_1']]) * (1 - params['True_P_1']) +
           (yy[t][states['F_AT_2']] + yy[t][states['F_ST_2']]) * (1 - params['True_P_2']) +
           params["True_N_1"] * (yy[t][states['F_NT_1']] + yy[t][states['F_AT_3']] +
                                 yy[t][states['F_FT_1']] + yy[t][states['F_GT1']] +
                                 yy[t][states['F_ST_4']] + yy[t][states['F_ST_3']] +
                                 yy[t][states['F_FT_3']]) +
           params["True_N_2"] * (yy[t][states['F_NT_2']] + yy[t][states['F_AT_4']] +
                                 yy[t][states['F_FT_2']] + yy[t][states['F_GT2']]) for t in range(N_t)]


    hospital = [sum(yy[t][states['F_H1']] + yy[t][states['F_H2']] + yy[t][states['F_H3']] for t in range(ti + 1)) for ti
                in range(N_t)]

    death = [yy[t][states['D']] for t in range(N_t)]

    return np.stack([pos, neg, hospital, death],axis=1)

In [ ]:
times = np.linspace(0, N_t, N_t * sample_num_factor+1)
duration=100

In [ ]:
def model(sim_par):
    pvec = [
        sim_par["beta"],
        sim_par["phi_s"],
        sim_par["g_rate"]
    ]
    sim = simulator(pvec)
    
    return {
            'cases1': poisson_noise(sim[:,0]),
            'cases2': poisson_noise(sim[:,1]),
            'cases3': poisson_noise(sim[:,2]),
            'cases4': poisson_noise(sim[:,3])
    }

In [ ]:
p_distance = pyabc.AdaptivePNormDistance(
    p=2,
    scale_function=pyabc.distance.std,
    all_particles_for_scale=True
)

In [ ]:
prior = Distribution(
    beta = RV("uniform", 0.6, 0.6),
    phi_s = RV("uniform", 0.2, 0.4),
    g_rate  = RV("uniform", 0.005, 0.005),
)

In [ ]:
eps = QuantileEpsilon(initial_epsilon='from_sample', alpha=0.2)
transition = MultivariateNormalTransition(scaling=0.5)

In [ ]:
results = []
gene_num = 100

for i, sim in enumerate(simulation_dataset):
    obs_data = sim['poisson']  
    obs_dict={"cases1": obs_data[:,0],
             "cases2": obs_data[:,1],
             "cases3": obs_data[:,2],
             "cases4": obs_data[:,3]}
 
    db_path = "sqlite:///" + tempfile.mkstemp(suffix=f"abc_{i}.db")[1]
    
    abc = ABCSMC(model, prior, p_distance, population_size=1000, transitions=transition, eps=eps)
    abc.new(db_path, obs_dict)

    history = abc.run(max_total_nr_simulations=100000)

    df_posterior = history.get_distribution()
    
    results.append({
        "params": sim['params'],
        "posterior": df_posterior,
        "populations": history.get_all_populations()
    })

In [ ]:
posterior_samples=[]
for i, res in enumerate(results):  
    df, weights = res['posterior']
    df = df[param_names]
    
    kde_estimator = pyabc.transition.MultivariateNormalTransition()
    kde_estimator.fit(df, weights)
    
    num_samples = 10000
    samples = kde_estimator.rvs(num_samples)
    posterior_samples.append(samples)